使用create_cistarget_motif_databases.py构建cisTarget数据库前的准备工作： 生成create_cistarget_motif_databases.py -f参数对应的fasta文件

In [1]:
%%bash

mkdir -p "./fasta"
wget -O "./fasta/Apis_cerana.fasta.gz" "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/029/169/275/GCF_029169275.2_AcerK_1.0/GCF_029169275.2_AcerK_1.0_genomic.fna.gz"
gunzip "./fasta/Apis_cerana.fasta.gz"

--2026-09-04 16:37:35--  https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/029/169/275/GCF_029169275.2_AcerK_1.0/GCF_029169275.2_AcerK_1.0_genomic.fna.gz
Resolving ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)... 2607:f220:41e:250::12, 2607:f220:41e:250::11, 130.14.250.7, ...
Connecting to ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)|2607:f220:41e:250::12|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 69251800 (66M) [application/x-gzip]
Saving to: './fasta/Apis_cerana.fasta.gz'

     0K .......... .......... .......... .......... ..........  0%  245K 4m36s
    50K .......... .......... .......... .......... ..........  0%  243K 4m37s
   100K .......... .......... .......... .......... ..........  0%  246K 4m36s
   150K .......... .......... .......... .......... ..........  0% 36.9M 3m27s
   200K .......... .......... .......... .......... ..........  0% 24.3M 2m46s
   250K .......... .......... .......... .......... ..........  0% 33.7M 2m19s
   300K .......... ....

Mu et al., 2025取的区域是: 基因起始位置(非狭义的TSS)上游5kb, 下游2kb

In [2]:
%%bash

awk '$0 !~ /^#/ && $3=="gene"' "./gtf/Apis_cerana.gtf" > "./gtf/Apis_cerana_genes.gtf" # 仅保留类型为gene的行, 且去掉注释行
bedtools sort -i "./gtf/Apis_cerana_genes.gtf" > "./gtf/Apis_cerana_genes_sorted.gtf" # 排序

In [ ]:
%%bash
#! 根据HY以往的操作记录(.subproject/01_choose_gene_in_cistarget/01_choose_gene_in_cistarget.py)和asertlab提供的feather，还需要去掉全部线粒体基因，使用剩余基因建库
# NC_014295.1 是线粒体对应的染色体名称
awk '$1 != "NC_014295.1"' "./gtf/Apis_cerana_genes_sorted.gtf" > "./gtf/Apis_cerana_genes_sorted_nomt.gtf"

In [6]:
%%bash

wc -l "./gtf/Apis_cerana_genes_sorted.gtf"
wc -l "./gtf/Apis_cerana_genes_sorted_nomt.gtf"
# 37个基因，未出错

12561 ./gtf/Apis_cerana_genes_sorted.gtf
12548 ./gtf/Apis_cerana_genes_sorted_nomt.gtf


# 开始提取上游5kb和下游2kb

### 读取GTF

In [10]:
import pyranges as pr

ac_gtf = pr.read_gtf("./gtf/Apis_cerana_genes_sorted_nomt.gtf").df
ac_gtf

,Chromosome,Source,Feature,Start,End,Score,Strand,Frame,gene_id,db_xref,description,gbkey,gene,gene_biotype,gene_synonym,partial
0,NC_083852.1,Gnomon,gene,19209,25539,.,+,.,LOC107992805,GeneID:107992805,MATH and LRR domain-containing protein PFE0570w,Gene,LOC107992805,protein_coding,NaN,NaN
1,NC_083852.1,Gnomon,gene,31867,34512,.,+,.,LOC107992868,GeneID:107992868,E3 ubiquitin-protein ligase RFWD3,Gene,LOC107992868,protein_coding,NaN,NaN
2,NC_083852.1,Gnomon,gene,38244,46847,.,+,.,LOC107992788,GeneID:107992788,ribose-phosphate pyrophosphokinase 1,Gene,LOC107992788,protein_coding,NaN,NaN
3,NC_083852.1,Gnomon,gene,50438,53323,.,+,.,LOC133667054,GeneID:133667054,"medium-chain specific acyl-CoA dehydrogenase, ...",Gene,LOC133667054,protein_coding,NaN,NaN
4,NC_083852.1,Gnomon,gene,53446,115364,.,+,.,LOC107992756,GeneID:107992756,nuclear hormone receptor FTZ-F1,Gene,LOC107992756,protein_coding,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12543,NW_026893559.1,Gnomon,gene,230125,230503,.,+,.,LOC133667677,GeneID:133667677,uncharacterized LOC133667677,Gene,LOC133667677,protein_coding,NaN,NaN
12544,NW_026893559.1,Gnomon,gene,52880,62536,.,-,.,LOC133667674,GeneID:133667674,uncharacterized LOC133667674,Gene,LOC133667674,protein_coding,NaN,NaN
12545,NW_026893559.1,Gnomon,gene,233705,238437,.,-,.,LOC133667678,GeneID:133667678,uncharacterized LOC133667678,Gene,LOC133667678,protein_coding,NaN,NaN
12546,NW_026893559.1,Gnomon,gene,239924,242142,.,-,.,LOC133667679,GeneID:133667679,uncharacterized LOC133667679,Gene,LOC133667679,protein_coding,NaN,NaN


### 添加各染色体长度(防止下游2kb超过染色体长度)

In [8]:
# 生成index
from pyfaidx import Fasta

Fasta("./fasta/Apis_cerana.fasta")

Fasta("./fasta/Apis_cerana.fasta")

In [9]:
import pandas as pd

ac_genome_length = pd.read_csv("./fasta/Apis_cerana.fasta.fai", header=None, sep="\t")[[0, 1]].set_index(0)[1]
ac_genome_length

0
NC_083852.1       27131825
NC_083853.1       15875455
NC_083854.1       13386827
NC_083855.1       13159592
NC_083856.1       13471485
NC_083857.1       17285085
NC_083858.1       13645534
NC_083859.1       12170773
NC_083860.1       11954044
NC_083861.1       12434583
NC_083862.1       15832348
NC_083863.1       11302906
NC_083864.1       11000381
NC_083865.1       11731789
NC_083866.1        9491825
NC_083867.1        7109830
NW_026893558.1     5207728
NW_026893559.1      250740
NW_026893560.1       93405
NW_026893561.1      504434
NC_141594.1          15890
Name: 1, dtype: int64

In [11]:
ac_gtf["chrom_length"] = ac_gtf["Chromosome"].map(ac_genome_length)
ac_gtf

,Chromosome,Source,Feature,Start,End,Score,Strand,Frame,gene_id,db_xref,description,gbkey,gene,gene_biotype,gene_synonym,partial,chrom_length
0,NC_083852.1,Gnomon,gene,19209,25539,.,+,.,LOC107992805,GeneID:107992805,MATH and LRR domain-containing protein PFE0570w,Gene,LOC107992805,protein_coding,NaN,NaN,27131825
1,NC_083852.1,Gnomon,gene,31867,34512,.,+,.,LOC107992868,GeneID:107992868,E3 ubiquitin-protein ligase RFWD3,Gene,LOC107992868,protein_coding,NaN,NaN,27131825
2,NC_083852.1,Gnomon,gene,38244,46847,.,+,.,LOC107992788,GeneID:107992788,ribose-phosphate pyrophosphokinase 1,Gene,LOC107992788,protein_coding,NaN,NaN,27131825
3,NC_083852.1,Gnomon,gene,50438,53323,.,+,.,LOC133667054,GeneID:133667054,"medium-chain specific acyl-CoA dehydrogenase, ...",Gene,LOC133667054,protein_coding,NaN,NaN,27131825
4,NC_083852.1,Gnomon,gene,53446,115364,.,+,.,LOC107992756,GeneID:107992756,nuclear hormone receptor FTZ-F1,Gene,LOC107992756,protein_coding,NaN,NaN,27131825
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12543,NW_026893559.1,Gnomon,gene,230125,230503,.,+,.,LOC133667677,GeneID:133667677,uncharacterized LOC133667677,Gene,LOC133667677,protein_coding,NaN,NaN,250740
12544,NW_026893559.1,Gnomon,gene,52880,62536,.,-,.,LOC133667674,GeneID:133667674,uncharacterized LOC133667674,Gene,LOC133667674,protein_coding,NaN,NaN,250740
12545,NW_026893559.1,Gnomon,gene,233705,238437,.,-,.,LOC133667678,GeneID:133667678,uncharacterized LOC133667678,Gene,LOC133667678,protein_coding,NaN,NaN,250740
12546,NW_026893559.1,Gnomon,gene,239924,242142,.,-,.,LOC133667679,GeneID:133667679,uncharacterized LOC133667679,Gene,LOC133667679,protein_coding,NaN,NaN,250740


In [15]:
sum(ac_gtf["chrom_length"].isna()) # 追加无误，继续

0

In [16]:
import numpy as np

plus = ac_gtf["Strand"] == "+"

ac_gtf["Neo_Start"] = np.where(plus, ac_gtf["Start"] - 5000, ac_gtf["End"] - 2001)
ac_gtf["Neo_End"]   = np.where(plus, ac_gtf["Start"] + 2001, ac_gtf["End"] + 5000)
ac_gtf

,Chromosome,Source,Feature,Start,End,Score,Strand,Frame,gene_id,db_xref,description,gbkey,gene,gene_biotype,gene_synonym,partial,chrom_length,Neo_Start,Neo_End
0,NC_083852.1,Gnomon,gene,19209,25539,.,+,.,LOC107992805,GeneID:107992805,MATH and LRR domain-containing protein PFE0570w,Gene,LOC107992805,protein_coding,NaN,NaN,27131825,14209,21210
1,NC_083852.1,Gnomon,gene,31867,34512,.,+,.,LOC107992868,GeneID:107992868,E3 ubiquitin-protein ligase RFWD3,Gene,LOC107992868,protein_coding,NaN,NaN,27131825,26867,33868
2,NC_083852.1,Gnomon,gene,38244,46847,.,+,.,LOC107992788,GeneID:107992788,ribose-phosphate pyrophosphokinase 1,Gene,LOC107992788,protein_coding,NaN,NaN,27131825,33244,40245
3,NC_083852.1,Gnomon,gene,50438,53323,.,+,.,LOC133667054,GeneID:133667054,"medium-chain specific acyl-CoA dehydrogenase, ...",Gene,LOC133667054,protein_coding,NaN,NaN,27131825,45438,52439
4,NC_083852.1,Gnomon,gene,53446,115364,.,+,.,LOC107992756,GeneID:107992756,nuclear hormone receptor FTZ-F1,Gene,LOC107992756,protein_coding,NaN,NaN,27131825,48446,55447
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12543,NW_026893559.1,Gnomon,gene,230125,230503,.,+,.,LOC133667677,GeneID:133667677,uncharacterized LOC133667677,Gene,LOC133667677,protein_coding,NaN,NaN,250740,225125,232126
12544,NW_026893559.1,Gnomon,gene,52880,62536,.,-,.,LOC133667674,GeneID:133667674,uncharacterized LOC133667674,Gene,LOC133667674,protein_coding,NaN,NaN,250740,60535,67536
12545,NW_026893559.1,Gnomon,gene,233705,238437,.,-,.,LOC133667678,GeneID:133667678,uncharacterized LOC133667678,Gene,LOC133667678,protein_coding,NaN,NaN,250740,236436,243437
12546,NW_026893559.1,Gnomon,gene,239924,242142,.,-,.,LOC133667679,GeneID:133667679,uncharacterized LOC133667679,Gene,LOC133667679,protein_coding,NaN,NaN,250740,240141,247142


#### 如果Neo_Start为负数，将其变成0，如果Neo_End超出了同行chrom_length，将其替换为chrom_length

In [18]:
ac_gtf["Neo_Start"] = ac_gtf["Neo_Start"].clip(lower=0)
ac_gtf["Neo_End"] = ac_gtf[["Neo_End", "chrom_length"]].min(axis=1)
ac_gtf

,Chromosome,Source,Feature,Start,End,Score,Strand,Frame,gene_id,db_xref,description,gbkey,gene,gene_biotype,gene_synonym,partial,chrom_length,Neo_Start,Neo_End
0,NC_083852.1,Gnomon,gene,19209,25539,.,+,.,LOC107992805,GeneID:107992805,MATH and LRR domain-containing protein PFE0570w,Gene,LOC107992805,protein_coding,NaN,NaN,27131825,14209,21210
1,NC_083852.1,Gnomon,gene,31867,34512,.,+,.,LOC107992868,GeneID:107992868,E3 ubiquitin-protein ligase RFWD3,Gene,LOC107992868,protein_coding,NaN,NaN,27131825,26867,33868
2,NC_083852.1,Gnomon,gene,38244,46847,.,+,.,LOC107992788,GeneID:107992788,ribose-phosphate pyrophosphokinase 1,Gene,LOC107992788,protein_coding,NaN,NaN,27131825,33244,40245
3,NC_083852.1,Gnomon,gene,50438,53323,.,+,.,LOC133667054,GeneID:133667054,"medium-chain specific acyl-CoA dehydrogenase, ...",Gene,LOC133667054,protein_coding,NaN,NaN,27131825,45438,52439
4,NC_083852.1,Gnomon,gene,53446,115364,.,+,.,LOC107992756,GeneID:107992756,nuclear hormone receptor FTZ-F1,Gene,LOC107992756,protein_coding,NaN,NaN,27131825,48446,55447
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12543,NW_026893559.1,Gnomon,gene,230125,230503,.,+,.,LOC133667677,GeneID:133667677,uncharacterized LOC133667677,Gene,LOC133667677,protein_coding,NaN,NaN,250740,225125,232126
12544,NW_026893559.1,Gnomon,gene,52880,62536,.,-,.,LOC133667674,GeneID:133667674,uncharacterized LOC133667674,Gene,LOC133667674,protein_coding,NaN,NaN,250740,60535,67536
12545,NW_026893559.1,Gnomon,gene,233705,238437,.,-,.,LOC133667678,GeneID:133667678,uncharacterized LOC133667678,Gene,LOC133667678,protein_coding,NaN,NaN,250740,236436,243437
12546,NW_026893559.1,Gnomon,gene,239924,242142,.,-,.,LOC133667679,GeneID:133667679,uncharacterized LOC133667679,Gene,LOC133667679,protein_coding,NaN,NaN,250740,240141,247142


### 因为需要使用bedtools getfasta -s，所以必须保留Strand列：

In [19]:
ac_gtf_4_bed = ac_gtf[["Chromosome", "Neo_Start", "Neo_End", "gene_id", "Score", "Strand"]]
ac_gtf_4_bed

,Chromosome,Neo_Start,Neo_End,gene_id,Score,Strand
0,NC_083852.1,14209,21210,LOC107992805,.,+
1,NC_083852.1,26867,33868,LOC107992868,.,+
2,NC_083852.1,33244,40245,LOC107992788,.,+
3,NC_083852.1,45438,52439,LOC133667054,.,+
4,NC_083852.1,48446,55447,LOC107992756,.,+
...,...,...,...,...,...,...
12543,NW_026893559.1,225125,232126,LOC133667677,.,+
12544,NW_026893559.1,60535,67536,LOC133667674,.,-
12545,NW_026893559.1,236436,243437,LOC133667678,.,-
12546,NW_026893559.1,240141,247142,LOC133667679,.,-


In [20]:
ac_gtf_4_bed.to_csv("./metadata/Acer_4_cisTarget.bed", sep="\t", index=False, header=False)

# 根据bed文件生成fasta

In [23]:
%%bash

bedtools getfasta \
    -fi "./fasta/Apis_cerana.fasta" \
    -bed "./metadata/Acer_4_cisTarget.bed" \
    -name \
    -s \
    -fo "./fasta/Apis_cerana_4_cisTarget.fasta"

In [ ]:
import pyfaidx

Am4cis = pyfaidx.Fasta("./fasta/Apis_cerana_4_cisTarget.fasta")
# 检查Apis_cerana_4_cisTarget.fasta.fai，确定长度是7001以保证bedtools没取错:

In [25]:
%%bash

head "./fasta/Apis_cerana_4_cisTarget.fasta.fai"

LOC107992805::NC_083852.1:14209-21210(+)	7001	42	7001	7002
LOC107992868::NC_083852.1:26867-33868(+)	7001	7086	7001	7002
LOC107992788::NC_083852.1:33244-40245(+)	7001	14130	7001	7002
LOC133667054::NC_083852.1:45438-52439(+)	7001	21174	7001	7002
LOC107992756::NC_083852.1:48446-55447(+)	7001	28218	7001	7002
LOC133667059::NC_083852.1:93406-100407(+)	7001	35263	7001	7002
LOC108004355::NC_083852.1:119392-126393(+)	7001	42309	7001	7002
LOC114576852::NC_083852.1:128607-135608(+)	7001	49355	7001	7002
LOC107992715::NC_083852.1:250011-257012(+)	7001	56401	7001	7002
LOC107992716::NC_083852.1:425494-432495(+)	7001	63447	7001	7002
